# Used Car Price Prediction: KNN

### Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

In [ ]:
cars_df = pd.read_csv( "new_used_car_v1.csv" )

In [ ]:
cars_df.sample(5)

In [ ]:
cars_df.info()

### Feature Set Selection

In [ ]:
cars_df.columns

In [ ]:
x_features = ['KM_Driven', 'Fuel_Type', 'age',
              'Transmission', 'Owner_Type', 'Seats', 
              'make', 'mileage_new', 'engine_new', 
              'power_new', 'Location']

In [ ]:
cat_features = ['Fuel_Type', 
                'Transmission', 'Owner_Type',
                'make', 'Location']

In [ ]:
num_features = list(set(x_features) - set(cat_features))

In [ ]:
num_features

In [ ]:
cars_df[x_features].info()

### Dropping Null Values

In [ ]:
cars_df = cars_df[x_features + ['Price']].dropna()

In [ ]:
cars_df.shape

In [ ]:
cars_df.sample(5)

### Setting X and y variables

In [ ]:
X = cars_df[x_features]
y = cars_df['Price']

### Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    train_size = 0.8,
                                                    random_state = 80)

In [ ]:
X_train

In [ ]:
X_train[0:10]

In [ ]:
X_test.shape

### Encode Categorical Variables

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ohe_encoder = OneHotEncoder(handle_unknown='ignore')

In [ ]:
ohe_encoder.fit(X_train[cat_features])

In [ ]:
ohe_encoder.categories_

In [ ]:
encoded_cat_names = list(ohe_encoder.get_feature_names(cat_features))
encoded_cat_names

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[        
        ('num', numeric_transformer, num_features),
        ('cat', ohe_encoder, cat_features)])

### KNN (K-Nearest Neighbor)


In [ ]:
sn.lmplot( data = cars_df.sample(100),
           x = 'power_new',
           y = 'Price');

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
#knn = KNeighborsRegressor(n_neighbors=20)
knn = KNeighborsRegressor(n_neighbors=20, weights='distance')

In [ ]:
knn_v1 = Pipeline(steps=[('preprocessor', preprocessor),
                          ('knn', knn)])

In [ ]:
knn_v1.fit(X_train, y_train)

In [ ]:
final_feature_names = num_features + list(knn_v1['preprocessor'].transformers_[1][1].get_feature_names())

### Predict on test set

In [ ]:
y_pred = knn_v1.predict(X_test)

In [ ]:
y_df = pd.DataFrame({"actual": y_test,
                     "predicted": y_pred,
                     "residual": y_pred - y_test})

In [ ]:
y_df.sample(10, random_state = 100)

### Measuring Accuracy

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
mse_v1 = mean_squared_error(y_test, y_pred)

In [ ]:
mse_v1

In [ ]:
rmse_v1 = np.sqrt(mse_v1)

In [ ]:
rmse_v1

In [ ]:
r2_score(y_test, y_pred)

### K Fold Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
scores = cross_val_score( knn_v1,
                          X_train,
                          y_train,
                          cv = 10,
                          scoring = 'r2')

In [ ]:
scores

In [ ]:
scores.mean()

In [ ]:
scores.std()

## Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
knn_params = { "knn__n_neighbors": [5, 10, 15, 20, 25],
               "knn__weights": ['uniform', 'distance'],
               "knn__metric": ['minkowski', 'euclidean']}

In [ ]:
knn_grid_v1 = GridSearchCV(knn_v1,
                           param_grid=knn_params,
                           cv = 10,
                           scoring = 'r2')

In [ ]:
knn_grid_v1.fit(X_train, y_train)

In [ ]:
knn_grid_v1.best_params_

In [ ]:
knn_grid_v1.best_score_

In [ ]:
pd.DataFrame( knn_grid_v1.cv_results_ )